# HiveMind-GNN: Neural Combinatorial Optimization for Bee-Worker Routing

*Exploring Graph Neural Networks for Multi-Agent Pathfinding*

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from src.env import FlowerFieldGenerator, HiveMindEnvironment
from src.model import HiveMindGNN, EdgePredictor

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Generate a Flower Field Graph

In [ ]:
generator = FlowerFieldGenerator(
    num_nodes=50,
    num_sources=5,
    num_sinks=5,
    density=0.3,
    seed=42
)

graph = generator.generate()
print(f"Graph nodes: {graph.number_of_nodes()}")
print(f"Graph edges: {graph.number_of_edges()}")

sources = [i for i, d in graph.nodes(data=True) if d.get('is_source', False)]
sinks = [i for i, d in graph.nodes(data=True) if d.get('is_sink', False)]
print(f"Sources: {sources}")
print(f"Sinks: {sinks}")

## 2. Visualize the Graph

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

pos = nx.spring_layout(graph, seed=42, k=2)

node_colors = []
for i in range(graph.number_of_nodes()):
    if graph.nodes[i].get('is_source', False):
        node_colors.append('#FFD700')  # Gold for sources
    elif graph.nodes[i].get('is_sink', False):
        node_colors.append('#FF6B6B')  # Red for sinks
    else:
        nectar = graph.nodes[i].get('nectar_density', 0)
        node_colors.append(plt.cm.YlOrRd(nectar))

node_sizes = [300 if graph.nodes[i].get('is_source', False) or graph.nodes[i].get('is_sink', False) else 100 for i in range(graph.number_of_nodes())]

nx.draw_networkx_nodes(graph, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8, ax=ax)
nx.draw_networkx_edges(graph, pos, edge_color='gray', alpha=0.4, arrows=True, ax=ax)

labels = {i: i for i in range(graph.number_of_nodes()) if graph.nodes[i].get('is_source', False) or graph.nodes[i].get('is_sink', False)}
nx.draw_networkx_labels(graph, pos, labels, font_size=8, font_weight='bold', ax=ax)

ax.set_title('Flower Field Graph\n(Yellow=Source, Red=Sink, Color=Nectar Density)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Node Features Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

nectar_levels = [graph.nodes[i].get('nectar_density', 0) for i in range(graph.number_of_nodes())]
occupancy = [graph.nodes[i].get('current_occupancy', 0) for i in range(graph.number_of_nodes())]
out_degree = [graph.out_degree(i) for i in range(graph.number_of_nodes())]
in_degree = [graph.in_degree(i) for i in range(graph.number_of_nodes())]

axes[0, 0].hist(nectar_levels, bins=20, color='#FFD700', edgecolor='black')
axes[0, 0].set_title('Nectar Density Distribution')
axes[0, 0].set_xlabel('Nectar Level')
axes[0, 0].set_ylabel('Count')

axes[0, 1].hist(out_degree, bins=15, color='#4ECDC4', edgecolor='black')
axes[0, 1].set_title('Out-Degree Distribution')
axes[0, 1].set_xlabel('Out-Degree')
axes[0, 1].set_ylabel('Count')

axes[1, 0].hist(in_degree, bins=15, color='#FF6B6B', edgecolor='black')
axes[1, 0].set_title('In-Degree Distribution')
axes[1, 0].set_xlabel('In-Degree')
axes[1, 0].set_ylabel('Count')

edge_weights = [generator.get_edge_weight(u, v, graph) for u, v in graph.edges()]
axes[1, 1].hist(edge_weights, bins=30, color='#95E1D3', edgecolor='black')
axes[1, 1].set_title('Edge Weight Distribution')
axes[1, 1].set_xlabel('Edge Weight')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 4. Environment Simulation

In [ ]:
env = HiveMindEnvironment(num_bees=10, graph_generator=generator)
obs = env.reset(graph)

print("Observation keys:", obs.keys())
print(f"Node features shape: {obs['node_features'].shape}")
print(f"Edge index shape: {obs['edge_index'].shape}")
print(f"Edge attributes shape: {obs['edge_attr'].shape}")

## 5. Random Policy Baseline

In [ ]:
total_reward, bees = env.simulate_random_policy()

print("Random Policy Results:")
print(f"Total Reward: {total_reward:.2f}")
print(f"Total Nectar Collected: {sum(b.nectar_collected for b in bees):.2f}")
print(f"Completed Bees: {sum(1 for b in bees if b.state.value == 'finished')}")
print()
for bee in bees[:3]:
    print(f"Bee {bee.bee_id}: {bee.nectar_collected:.2f} nectar, {bee.steps_taken} steps")

## 6. Initialize and Test GNN Model

In [ ]:
import torch

model = HiveMindGNN(
    node_input_dim=7,
    edge_input_dim=2,
    hidden_dim=64,
    num_layers=3
)

node_features = torch.tensor(obs['node_features'])
edge_index = torch.tensor(obs['edge_index'])
edge_attr = torch.tensor(obs['edge_attr'])

embeddings, edge_logits = model(node_features, edge_index, edge_attr)

print(f"Node embeddings shape: {embeddings.shape}")
print(f"Edge logits shape: {edge_logits.shape}")
print(f"Sample edge logits: {edge_logits[:5].detach().numpy()}")

## 7. Edge Probability Visualization

In [ ]:
edge_probs = torch.sigmoid(edge_logits).detach().numpy()

fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(range(len(edge_probs)), np.sort(edge_probs)[::-1], color='#6C5CE7', alpha=0.7)
ax.axhline(y=np.mean(edge_probs), color='red', linestyle='--', label=f'Mean: {np.mean(edge_probs):.3f}')
ax.set_xlabel('Edge Index (sorted)')
ax.set_ylabel('Probability of Being Optimal')
ax.set_title('GNN Predicted Edge Importance Scores')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Sample Graphs Generation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx in range(6):
    gen = FlowerFieldGenerator(num_nodes=30, seed=idx*10)
    g = gen.generate()
    
    pos = nx.spring_layout(g, seed=42, k=1.5)
    
    node_colors = []
    for i in range(g.number_of_nodes()):
        if g.nodes[i].get('is_source', False):
            node_colors.append('#FFD700')
        elif g.nodes[i].get('is_sink', False):
            node_colors.append('#FF6B6B')
        else:
            node_colors.append('#A8E6CF')
    
    nx.draw_networkx_nodes(g, pos, node_color=node_colors, node_size=50, ax=axes[idx])
    nx.draw_networkx_edges(g, pos, edge_color='gray', alpha=0.3, ax=axes[idx])
    axes[idx].set_title(f'Flower Field #{idx+1}\n({g.number_of_nodes()} nodes, {g.number_of_edges()} edges)')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrates:
- **Graph Generation**: Creating dynamic flower field graphs with variable nectar levels
- **Environment Simulation**: Multi-agent bee routing with congestion and reward mechanisms
- **GNN Processing**: Node embedding and edge scoring through Graph Attention Networks
- **Visualization**: Understanding graph structure, node features, and model predictions

Next steps: Train the model on the generated dataset to learn optimal routing policies!